# 05_silver_payment_csv
-Bronze -> Silver for `payment` (internal CSV, 21 rows - the internal
ledger, NOT the external PAYMENT_*.dat feed - that's 10_silver_external_payment).



In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F
from datetime import date

SOURCE_NAME = "payment"
REQUIRED_COLS = ["payment_id", "commitment_id", "fund_id", "investor_id", "payment_type", "amount_usd", "status", "event_date"]
KEY_COLS = ["payment_id"]
COMPARE_COLS = ["commitment_id", "fund_id", "investor_id", "amount_usd", "status", "event_date"]
business_date_str = date.today().isoformat()

In [0]:
bronze_df = read_bronze(spark, SOURCE_NAME)
print(f"Bronze row count: {bronze_df.count()}")

Bronze row count: 222


In [0]:
typed_df = (
    bronze_df
    .withColumn("payment_id", F.trim(F.col("payment_id")))
    .withColumn("commitment_id", F.trim(F.col("commitment_id")))
    .withColumn("fund_id", F.trim(F.col("fund_id")))
    .withColumn("investor_id", F.trim(F.col("investor_id")))
    .withColumn("company_id", F.trim(F.col("company_id")))
    .withColumn("payment_type", F.upper(F.trim(F.col("payment_type"))))
    .withColumn("amount_usd", F.col("amount_usd").cast("double"))
    .withColumn("status", F.upper(F.trim(F.col("status"))))
    .withColumn("event_date", F.to_date("event_date"))
    .withColumn("entry_benchmark_price", F.col("entry_benchmark_price").cast("double"))
)

In [0]:
clean_df, null_rejects_df = split_on_required_nulls(typed_df, REQUIRED_COLS)
null_reject_count = null_rejects_df.count()
if null_reject_count > 0:
    write_quarantine(null_rejects_df, SOURCE_NAME)

### Expected-pattern check (not a hard reject, just visibility)
Confirm company_id/entry_benchmark_price are only populated for
INVESTMENT rows - and flag (don't reject) any INVESTMENT row that's
MISSING them, since that IS unexpected.

In [0]:
unexpected_investment_gaps = clean_df.filter(
    (F.col("payment_type") == "INVESTMENT") &
    (F.col("company_id").isNull() | F.col("entry_benchmark_price").isNull())
)
unexpected_gap_count = unexpected_investment_gaps.count()
print(f"INVESTMENT rows missing company_id/entry_benchmark_price (unexpected): {unexpected_gap_count}")
if unexpected_gap_count > 0:
    write_quarantine(
        unexpected_investment_gaps.withColumn("reason_code", F.lit("INVESTMENT_MISSING_LINK")),
        SOURCE_NAME
    )

INVESTMENT rows missing company_id/entry_benchmark_price (unexpected): 8


### Referential integrity

In [0]:
fund_silver = spark.table(silver_table("fund"))
investor_silver = spark.table(silver_table("investor"))
commitment_silver = spark.table(silver_table("commitment"))
company_silver = spark.table(silver_table("portfolio_company"))

valid_fk_df, invalid_fund_fk = check_foreign_key(clean_df, "fund_id", fund_silver, "fund_id")
valid_fk_df, invalid_investor_fk = check_foreign_key(valid_fk_df, "investor_id", investor_silver, "investor_id")
valid_fk_df, invalid_commitment_fk = check_foreign_key(valid_fk_df, "commitment_id", commitment_silver, "commitment_id")
# company_id is nullable - only check FK where it's populated
valid_fk_df, invalid_company_fk = check_foreign_key(valid_fk_df, "company_id", company_silver, "company_id")

invalid_fk_count = (
    invalid_fund_fk.count() + invalid_investor_fk.count() +
    invalid_commitment_fk.count() + invalid_company_fk.count()
)
for fk_df in [invalid_fund_fk, invalid_investor_fk, invalid_commitment_fk, invalid_company_fk]:
    if fk_df.count() > 0:
        write_quarantine(fk_df, SOURCE_NAME)

In [0]:
deduped_df, duplicates_df, breaks_df = split_duplicates(valid_fk_df, KEY_COLS, COMPARE_COLS)
dup_count = duplicates_df.count()
break_count = breaks_df.count()
if dup_count > 0:
    write_quarantine(duplicates_df, SOURCE_NAME)
if break_count > 0:
    write_quarantine(breaks_df.withColumn("reason_code", F.lit("PAYMENT_ATTRIBUTE_BREAK")), SOURCE_NAME)

In [0]:
write_silver(deduped_df, SOURCE_NAME)
print(f"Silver row count: {deduped_df.count()}")

Silver row count: 222


In [0]:
log_dq(spark, SOURCE_NAME, business_date_str, "null_required_field", bronze_df.count(), null_reject_count, "NULL_REQUIRED_FIELD")
log_dq(spark, SOURCE_NAME, business_date_str, "unexpected_investment_gap", clean_df.count(), unexpected_gap_count, "INVESTMENT_MISSING_LINK")
log_dq(spark, SOURCE_NAME, business_date_str, "unknown_reference", clean_df.count(), invalid_fk_count, "UNKNOWN_REFERENCE")
log_dq(spark, SOURCE_NAME, business_date_str, "duplicate_record", valid_fk_df.count(), dup_count, "DUPLICATE_RECORD")
log_dq(spark, SOURCE_NAME, business_date_str, "attribute_break", valid_fk_df.count(), break_count, "PAYMENT_ATTRIBUTE_BREAK")

/home/spark-f5a9e09f-0021-42c0-b4e9-eb/.ipykernel/71/command-5696143635338729-2886423099:181: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


In [0]:
bronze_count = bronze_df.count()
silver_count = deduped_df.count()
quarantined_count = null_reject_count + invalid_fk_count + dup_count
assert bronze_count == silver_count + quarantined_count, (
    f"Row count mismatch: bronze={bronze_count}, silver={silver_count}, quarantined={quarantined_count}"
)
print(f"OK: bronze={bronze_count} = silver={silver_count} + quarantined={quarantined_count}")
print(f"(of which, {unexpected_gap_count} Silver rows are also flagged INVESTMENT_MISSING_LINK for review)")

OK: bronze=222 = silver=222 + quarantined=0
(of which, 8 Silver rows are also flagged INVESTMENT_MISSING_LINK for review)
